# Chapter 1 — Vectors and Linear Combinations

Companion notebook: every figure and timing in the chapter is produced here.

Reproduce with `clae-py -m nbconvert --to notebook --execute --inplace ch01.ipynb`.

## 1.0 — axpy: a pure-Python list comprehension vs NumPy (Listings 1.1–1.2, Figure 1.1)

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def list_comp_in_python(a: float, x: np.ndarray, y: np.ndarray) -> list:
    return [a * xi + yi for xi, yi in zip(x, y)]


def vectorized_in_numpy(a: float, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    return a * x + y

In [3]:
a = 2.5
rng = np.random.default_rng(0)
x, y = rng.random(10_000_000), rng.random(10_000_000)

t0 = time.perf_counter(); list_comp_in_python(a, x, y)
t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); vectorized_in_numpy(a, x, y)
t_vec = time.perf_counter() - t0

print(f'list comprehension: {t_loop:5.2f} s')
print(f'vectorized:         {t_vec * 1e3:5.0f} ms')
print(f'factor:             {t_loop / t_vec:5.0f}x')

list comprehension:  6.09 s
vectorized:            79 ms
factor:                77x


In [4]:
def best(fn, a, x, y, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(a, x, y); ts.append(time.perf_counter() - t0)
    return min(ts)

ns = np.logspace(3, 7, 9).astype(int)
t_loop = [best(list_comp_in_python, a, rng.random(n), rng.random(n), 1 if n > 10**6 else 3) for n in ns]
t_vec = [best(vectorized_in_numpy, a, rng.random(n), rng.random(n), 7) for n in ns]

plt.figure(figsize=(7, 5))
plt.semilogx(ns, t_loop, 'o-', label='list_comp_in_python (list comprehension)')
plt.semilogx(ns, t_vec, 's-', label='vectorized_in_numpy')
plt.xlabel('n  (length of x and y)'); plt.ylabel('wall-clock time (s)')
plt.title('axpy: scale a vector and add another, loop vs vectorized')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/fig_axpy_timing.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.2 — The claim on the table (Listings 1.5–1.6, the 10-house table, Figure 1.4)

In [5]:
import pandas as pd

zoning  = pd.read_csv('data/zoning.csv')
listing = pd.read_csv('data/listing.csv')
sale    = pd.read_csv('data/sale.csv')          # SalePrice (the target) lives here
housing = pd.merge(zoning, listing, on='Id')
housing = pd.merge(housing, sale, on='Id').set_index('Id')
housing.shape                                    # 1460 homes, 80 features

(1460, 80)

In [6]:
X = housing[['GrLivArea', 'OverallQual']].to_numpy(float)
y = housing['SalePrice'].to_numpy(float)

w, *_ = np.linalg.lstsq(X, y, rcond=None)
print('w:', np.round(w, 2))

# house 2, worked term by term (the align in Section 1.2)
gr, oq = X[1]
print(f'{w[0]:.2f} * {gr:.0f} + {w[1]:.2f} * {oq:.0f} '
      f'= {w[0]*gr:,.0f} + {w[1]*oq:,.0f} = {w[0]*gr + w[1]*oq:,.0f}')
print(f'actual: {y[1]:,.0f}')

w: [   51.87 17604.21]
51.87 * 1262 + 17604.21 * 6 = 65,460 + 105,625 = 171,085
actual: 181,500


In [7]:
pred = X @ w
first_ten = pd.DataFrame({'GrLivArea': X[:10, 0].astype(int),
                          'OverallQual': X[:10, 1].astype(int),
                          'predicted': pred[:10].round(0).astype(int),
                          'actual': y[:10].astype(int)},
                         index=housing.index[:10])
first_ten

,GrLivArea,OverallQual,predicted,actual
Id,,,,
1,1710,7,211927,208500
2,1262,6,171085,181500
3,1786,7,215869,223500
4,1717,7,212290,140000
5,2198,8,254844,250000
6,1362,5,158668,143000
7,1694,8,228701,307000
8,2090,7,231637,200000
9,1774,7,215247,129900


In [8]:
plt.figure(figsize=(6.5, 6.5))
plt.scatter(pred, y, s=8, alpha=0.35)
lim = [0, 800_000]
plt.plot(lim, lim, color='tab:red', lw=1, label='predicted = actual')
plt.xlim(lim); plt.ylim(lim); plt.gca().set_aspect('equal')
plt.xlabel('predicted price ($)'); plt.ylabel('actual sale price ($)')
plt.title('One axpy pass prices 1,460 homes')
plt.grid(True, alpha=0.3); plt.legend()
plt.savefig('figures/fig_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.1 — Two operations, drawn (Listings 1.3–1.4, Figures 1.2–1.3)

In [9]:
def plot_vector(v, color='blue', label=None, **kw):
    plt.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
               color=color, label=label, **kw)

### Scalar multiplication: scaling along the line

In [10]:
v = np.array([2, 1])

plt.figure(figsize=(6, 6))
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
t = np.array([-1.6, 2.3])
plt.plot(t * v[0], t * v[1], '--', color='gray', alpha=0.5)   # the line through the origin
plot_vector(2 * v, 'tab:purple', '2v')
plot_vector(v, 'tab:blue', 'v')
plot_vector(-v, 'tab:red', '-v')
plt.xlim(-4, 5); plt.ylim(-2.5, 3); plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3); plt.legend(); plt.title('Scalar multiplication')
plt.savefig('figures/fig_scalar_multiplication.png', dpi=150, bbox_inches='tight')
plt.show()

### Vector addition: tip to tail

In [11]:
v1 = np.array([1, 2]); v2 = np.array([3, 1])

plt.figure(figsize=(6, 6))
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plot_vector(v1, 'tab:blue', 'v1')
plot_vector(v2, 'tab:red', 'v2')
plot_vector(v1 + v2, 'tab:green', 'v1 + v2')
plt.quiver(v1[0], v1[1], v2[0], v2[1], angles='xy', scale_units='xy',
           scale=1, color='tab:red', alpha=0.35)
plt.xlim(-1, 5); plt.ylim(-1, 4); plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3); plt.legend(); plt.title('Vector addition: tip to tail')
plt.savefig('figures/fig_vector_addition.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.3 — Span, swept by meshgrid (Listing 1.7, Figure 1.5)

In [12]:
def span_cloud(v: np.ndarray, w: np.ndarray, ax) -> None:
    Cc, Dd = np.meshgrid(np.linspace(-2, 2, 25), np.linspace(-2, 2, 25))
    cloud = Cc.ravel()[:, None] * v + Dd.ravel()[:, None] * w
    ax.scatter(cloud[:, 0], cloud[:, 1], s=4, alpha=0.4)

v = np.array([2, 1])
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 5))
for ax, w2, title in [(left, 2 * v, 'w = 2v: the sweep collapses to a line'),
                      (right, np.array([1, 3]), 'w off the line: the sweep fills a plane')]:
    span_cloud(v, w2, ax)
    ax.quiver(0, 0, *v, angles='xy', scale_units='xy', scale=1, color='tab:blue')
    ax.quiver(0, 0, *w2, angles='xy', scale_units='xy', scale=1, color='tab:red')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.set_title(title, fontsize=10)
plt.savefig('figures/fig_span_pair.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 — Independence: one of each verdict, and the membership cloud (Figure 1.6)

In [13]:
v = np.array([2, 1]); w = np.array([1, 3]); u = np.array([4, 7])
print('independent pair: only the zero combination ->', 0 * v + 0 * w)
print('dependent triple: 1v + 2w - 1u =', 1 * v + 2 * w - 1 * u)

independent pair: only the zero combination -> [0 0]
dependent triple: 1v + 2w - 1u = [0 0]


In [14]:
plt.figure(figsize=(6, 6))
ax = plt.gca()
span_cloud(v, w, ax)
for vec, c, name in [(v, 'tab:blue', 'v'), (w, 'tab:red', 'w'), (u, 'tab:green', 'u')]:
    ax.quiver(0, 0, *vec, angles='xy', scale_units='xy', scale=1, color=c, label=name)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title('u = 1v + 2w lands inside the span: the triple is dependent')
plt.savefig('figures/fig_span_membership.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 — Magnitude, direction, the dot product (Listings 1.8–1.9)

In [15]:
v = np.array([3, 1]); w = np.array([2, 3])
score = v @ w
agreement = score / (np.linalg.norm(v) * np.linalg.norm(w))
print(score, round(agreement, 3), round(np.degrees(np.arccos(agreement))))

9 0.789 38


In [16]:
import pandas as pd

zoning = pd.read_csv('data/zoning.csv')
listing = pd.read_csv('data/listing.csv')
sale = pd.read_csv('data/sale.csv')
housing = pd.merge(zoning, listing, on='Id')
housing = pd.merge(housing, sale, on='Id').set_index('Id')

a = housing['GrLivArea'].to_numpy(float)
b = housing['OverallQual'].to_numpy(float)
lhs = abs(a @ b)
rhs = np.linalg.norm(a) * np.linalg.norm(b)
print(f'|a.b|     = {lhs:,.0f}')
print(f'|a||b|    = {rhs:,.0f}')
print(f'lhs/rhs   = {lhs/rhs:.4f}   (must be <= 1)')

|a.b|     = 14,123,976
|a||b|    = 14,645,262
lhs/rhs   = 0.9644   (must be <= 1)
